# 第15周 · 自编码器与 VAE（复现报告）

**本周结束时你能做到**：写出一个自编码器并画出它的 2 维潜空间；用实验说明"线性自编码器 ≈ PCA"；解释 VAE 的编码器为什么输出 $\mu$ 与 $\sigma$、为什么需要重参数化技巧、ELBO = 重建项 − KL 项各自在做什么；从潜空间采样生成新图、在两张图之间插值；把"选哪个似然（损失函数）"当成一次受控实验来做；认出后验坍塌，并知道 KL 退火这个补救办法。

上周的数学工作坊是本周的直接前置：**交叉熵 = 负对数似然**（VAE 的重建项），以及 $\mathrm{KL}(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0, 1)) = -\frac{1}{2}(1 + \log\sigma^2 - \mu^2 - \sigma^2)$（VAE 的 KL 项）。

本周蓝本：

- VAE 代码：PyTorch 官方示例 [pytorch/examples/vae](https://github.com/pytorch/examples/tree/main/vae)（BSD-3-Clause）——MNIST、784→400→(μ, logvar 各 20 维)、重参数化、解码 20→400→784 + sigmoid、损失 = BCE(sum) + KL、Adam 1e-3。我们几乎逐行照搬，只把数据换成一次算好的张量。
- 理论：Kingma & Welling 2013, *Auto-Encoding Variational Bayes*（[arXiv:1312.6114](https://arxiv.org/abs/1312.6114)）；β-VAE：Higgins et al., ICLR 2017；后验坍塌与 KL 退火：Bowman et al. 2016（[arXiv:1511.06349](https://arxiv.org/abs/1511.06349)）；线性自编码器与 PCA：Baldi & Hornik 1989。
- 自编码器部分的叙事借自 [UvA DL Tutorial 9: Deep Autoencoders](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial9/AE_CIFAR10.html)（Phillip Lippe，MIT）：潜维度消融、"自编码器不是生成模型"。原教程用 CIFAR-10 + 卷积网络 + MSE，我们换成 MNIST + 全连接网络（与 pytorch/examples 同骨架，也能和第 6 周的 PCA 直接比较），**只借叙事、不照搬数据与代码**。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from zero2researcher import data, experiments
from zero2researcher.checks import week15 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_dataset(cls, train):
    # 下载并缓存（MNIST / FashionMNIST 各约 30MB，只有第一次要等）；网络抖动时自动重试一次
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


def to_pixels(ds):
    # uint8 图像 (N, 28, 28) → [0, 1] 的 float 张量 (N, 784)。不做标准化：BCE 要求目标在 [0, 1] 之间
    return ds.data.float().div(255).reshape(len(ds), -1), ds.targets


mnist_train, mnist_test = load_dataset(datasets.MNIST, True), load_dataset(datasets.MNIST, False)
X_all, y_all = to_pixels(mnist_train)
X_test, y_test = to_pixels(mnist_test)
X_train, y_train = X_all[:10000], y_all[:10000]          # 本周讲义统一的训练集：前 1 万张
print("训练", tuple(X_train.shape), "| 测试", tuple(X_test.shape), "| 像素范围", X_train.min().item(), "–", X_train.max().item())


def show_images(images, nrow, title, ax=None):
    # 把 (N, 784) 的图拼成 nrow 行的网格画出来
    images = images.detach().cpu().reshape(-1, 28, 28)
    ncol = int(np.ceil(len(images) / nrow))
    grid = np.zeros((nrow * 29, ncol * 29))          # 图与图之间留 1 像素白边
    for i, img in enumerate(images):
        r, c = divmod(i, ncol)
        grid[r * 29:r * 29 + 28, c * 29:c * 29 + 28] = img.numpy()
    ax = ax or plt.subplots(figsize=(ncol * 0.6, nrow * 0.6))[1]
    ax.imshow(grid, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

---
## S1 · 自编码器

**自编码器**（autoencoder）由两半组成：**编码器**把一张 784 维的图压成一个很短的向量 $z$（潜表示，latent），**解码器**再从 $z$ 把图还原出来。训练目标只有一个：还原得越像越好。输入就是标签——不需要任何人工标注，这是一种**自监督**学习（第 13 周的掩码语言模型是它的近亲）。

$$x \xrightarrow{\ \text{encoder}\ } z \in \mathbb{R}^{k} \xrightarrow{\ \text{decoder}\ } \hat{x} \approx x$$

关键在**瓶颈**：$k$ 远小于 784，网络被迫只保留"最重要的信息"。先取 $k = 2$——这样潜空间可以直接画出来。

重建损失用 BCE（上周 S3：伯努利负对数似然），解码器最后接 sigmoid，输出每个像素"为黑"的概率；按 pytorch/examples 的口径对像素**求和**，再除以 batch 大小，得到"每张图"的损失。

In [ ]:
def make_autoencoder(k):
    # 784 → 256 → k → 256 → 784（最后 sigmoid 把输出压进 [0, 1]）
    encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, k))
    decoder = nn.Sequential(nn.Linear(k, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    return encoder, decoder


def train_autoencoder(k, epochs=10, seed=0, X=X_train):
    experiments.set_seed(seed)
    encoder, decoder = make_autoencoder(k)
    encoder, decoder = encoder.to(device), decoder.to(device)
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
    loader = DataLoader(TensorDataset(X), batch_size=128, shuffle=True)
    for _ in range(epochs):
        for (xb,) in loader:
            xb = xb.to(device)
            loss = F.binary_cross_entropy(decoder(encoder(xb)), xb, reduction="sum") / len(xb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return encoder.eval(), decoder.eval()


def recon_mse(encoder, decoder, X):
    # 每个像素的平均平方误差（和 PCA 比较时用同一个口径）
    with torch.no_grad():
        return F.mse_loss(decoder(encoder(X.to(device))), X.to(device)).item()


t0 = time.time()
enc2, dec2 = train_autoencoder(k=2)
print(f"k = 2 的自编码器：测试集每像素 MSE {recon_mse(enc2, dec2, X_test):.4f}（{time.time() - t0:.1f} 秒）")

In [ ]:
with torch.no_grad():
    z2 = enc2(X_test[:3000].to(device)).cpu().numpy()
    recon = dec2(enc2(X_test[:10].to(device)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
sc = axes[0].scatter(z2[:, 0], z2[:, 1], c=y_test[:3000], cmap="tab10", s=4)
fig.colorbar(sc, ax=axes[0], ticks=range(10), label="Digit")
axes[0].set_xlabel("z1")
axes[0].set_ylabel("z2")
axes[0].set_title("A 2-D bottleneck already groups digits, but the clusters overlap")
show_images(torch.cat([X_test[:10], recon.cpu()]), nrow=2, title="Top: test digits, bottom: reconstructions (k = 2)", ax=axes[1])
plt.tight_layout()
plt.show()

没有用到任何标签，同一个数字的图却自己聚在了一起（0、1 很清楚，4/9、3/5/8 挤在一起）。只有 2 个数的瓶颈，重建只保留了粗略的形状：很多数字被还原成了"长得像的另一个"（4 变成 9、2 变成 3），细节都糊了。

### 潜维度越大，重建越好

UvA Tutorial 9 的第一个实验：只改瓶颈宽度 $k$，看重建误差。

In [ ]:
ks = [2, 8, 32]
aes = {2: (enc2, dec2)}
for k in ks[1:]:
    aes[k] = train_autoencoder(k)
ae_mse = {k: recon_mse(*aes[k], X_test) for k in ks}
for k in ks:
    print(f"k = {k:2d}: 测试集每像素 MSE {ae_mse[k]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={"width_ratios": [1, 2]})
axes[0].plot(ks, [ae_mse[k] for k in ks], "o-")
axes[0].set_xscale("log", base=2)
axes[0].set_xticks(ks, [str(k) for k in ks])
axes[0].set_xlabel("Latent size k")
axes[0].set_ylabel("Test MSE per pixel")
axes[0].set_title("A wider bottleneck reconstructs better")
with torch.no_grad():
    rows = [X_test[10:20]] + [aes[k][1](aes[k][0](X_test[10:20].to(device))).cpu() for k in ks]
show_images(torch.cat(rows), nrow=4, title="Rows: original, k = 2, 8, 32", ax=axes[1])
plt.tight_layout()
plt.show()

### 自编码器不是生成模型

能不能把解码器当"画图机器"——随便给一个 $z$，画出一个新数字？试试：在 $k = 32$ 的潜空间里随机抽 16 个点（每个维度按训练集潜向量的均值和标准差抽），交给解码器：

In [ ]:
enc32, dec32 = aes[32]
with torch.no_grad():
    z_train = enc32(X_train.to(device))
    torch.manual_seed(0)
    z_rand = z_train.mean(0) + z_train.std(0) * torch.randn(16, 32, device=device)
    fake = dec32(z_rand)
show_images(fake, nrow=2, title="Decoding random latent points of a plain autoencoder: mostly not digits")
plt.show()

大多数不像数字。原因是：自编码器只被要求"把训练图编码、再解码回来"，**没有人规定潜空间的其余部分长什么样**。训练图的编码在潜空间里散成一些不规则的团块，团块之间的"空地"解码器从没见过，画出什么都有可能。下一节的 VAE 就是来修这件事的。

### 线性自编码器 ≈ PCA

把编码器和解码器里的非线性全部去掉，只剩两个线性层 $784 \to k \to 784$，用 MSE 训练——这就是**线性自编码器**。Baldi & Hornik（1989）证明：它的误差面上没有坏的局部极小，最优解恰好把数据投影到**前 $k$ 个主成分张成的子空间**——也就是第 6 周的 PCA。用实验验证：同样 $k = 16$、同一份 1 万张训练图，比较两者在测试集上的重建误差。

In [ ]:
K_LIN = 16
experiments.set_seed(0)
linear_ae = nn.Sequential(nn.Linear(784, K_LIN), nn.Linear(K_LIN, 784)).to(device)   # 没有激活函数
optimizer = torch.optim.Adam(linear_ae.parameters(), lr=3e-3)
loader = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
for _ in range(20):
    for (xb,) in loader:
        xb = xb.to(device)
        loss = F.mse_loss(linear_ae(xb), xb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
with torch.no_grad():
    ae_err = F.mse_loss(linear_ae(X_test.to(device)), X_test.to(device)).item()

pca = PCA(n_components=K_LIN).fit(X_train.numpy())
pca_recon = pca.inverse_transform(pca.transform(X_test.numpy()))
pca_err = float(np.mean((pca_recon - X_test.numpy()) ** 2))
nonlinear_err = recon_mse(*train_autoencoder(K_LIN), X_test)
print(f"k = {K_LIN}：线性 AE {ae_err:.5f} | PCA {pca_err:.5f}（比值 {ae_err / pca_err:.3f}）| 非线性 AE {nonlinear_err:.5f}")
ck.linear_ae_vs_pca(ae_err, pca_err)

线性 AE 与 PCA 的误差几乎一样（PCA 是理论最优，线性 AE 只能无限接近）；加上非线性之后误差明显更低——这就是"深度"自编码器的意义。

还可以验证得更彻底：线性 AE 解码器的 16 列权重张成的子空间，和 PCA 前 16 个主成分张成的子空间是不是同一个？两个子空间之间的"主角余弦"全为 1 就说明完全重合（线性 AE 的权重本身不必是正交的主成分，只要张成同一个子空间）：

In [ ]:
W = linear_ae[1].weight.detach().cpu().numpy()          # 解码器权重 (784, 16)
Q, _ = np.linalg.qr(W)                                   # 正交化
cosines = np.linalg.svd(Q.T @ pca.components_.T, compute_uv=False)
print("两个子空间的主角余弦:", np.round(cosines, 3))

---
## S2 · 变分自编码器（VAE）

### 从"一个点"到"一团分布"

VAE（Kingma & Welling 2013）对自编码器做了两处改动：

1. **编码器输出一个分布**，而不是一个点：对每张图 $x$，给出一个高斯 $q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$——实际输出 $\mu$ 和 $\log\sigma^2$（上周讲过：用 `logvar` 可以取任意实数）。训练时从这个分布里**抽一个 $z$** 交给解码器；
2. **损失里加一项 KL**，要求每张图的分布都别离标准正态 $p(z) = \mathcal{N}(0, I)$ 太远：

$$\mathcal{L}(x) = \underbrace{\mathbb{E}_{z \sim q(z|x)}\big[-\log p(\hat{x} = x \mid z)\big]}_{\text{重建项：BCE}} \;+\; \underbrace{\mathrm{KL}\big(q(z|x)\,\|\,\mathcal{N}(0, I)\big)}_{\text{KL 项：} -\frac{1}{2}\sum_j (1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)}$$

两项在拔河：重建项想让每张图的编码精确、彼此分开（$\sigma$ 越小越好）；KL 项想把所有编码都拉到原点附近、并保持一定的"宽度"。结果是：**潜空间被训练图的分布填满、没有空地**——从 $\mathcal{N}(0, I)$ 随便抽一个 $z$，解码出来都应该像一张图。这就修好了 S1 的问题。

**这个损失从哪来（直观版）**：我们真正想要的是让数据的似然 $\log p(x)$ 尽量大，但它要对所有 $z$ 积分，算不出来。可以证明

$$\log p(x) = \underbrace{\mathbb{E}_{q}[\log p(x|z)] - \mathrm{KL}(q(z|x)\,\|\,p(z))}_{\text{ELBO（证据下界）}} + \underbrace{\mathrm{KL}\big(q(z|x)\,\|\,p(z|x)\big)}_{\ge 0}$$

最后一项是 KL，永远 ≥ 0（上周的性质 1），所以 ELBO 是 $\log p(x)$ 的一个**下界**：把 ELBO 推高，就是在推高似然。ELBO = 重建 − KL；上面的 $\mathcal{L}$ 就是 $-\text{ELBO}$。

### 为什么需要重参数化

训练时要对"从 $q(z|x)$ 抽一个 $z$"这一步求导——梯度得从解码器穿过 $z$ 传回编码器的 $\mu$ 和 $\sigma$。可是**"抽样"这个操作没有导数**：

In [ ]:
mu = torch.tensor([0.5, -1.0], requires_grad=True)
logvar = torch.tensor([0.0, 0.0], requires_grad=True)
z = torch.distributions.Normal(mu, torch.exp(0.5 * logvar)).sample()     # 直接从 N(mu, sigma²) 抽样
print("抽到的 z:", z, "| 能求导吗:", z.requires_grad)

In [ ]:
z.sum().backward()     # 报错：抽样的结果和 mu、logvar 之间没有计算图

**重参数化技巧**（论文摘要里的 "reparameterization of the variational lower bound"）：把随机性挪到一个**与参数无关**的噪声 $\varepsilon$ 上，

$$z = \mu + \sigma \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I), \qquad \sigma = \exp(\tfrac{1}{2}\log\sigma^2)$$

$z$ 的分布没变（还是 $\mathcal{N}(\mu, \sigma^2)$），但现在它是 $\mu$ 和 $\sigma$ 的普通函数——加法和乘法都能求导，$\partial z / \partial \mu = 1$、$\partial z / \partial \sigma = \varepsilon$：

In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


z = reparameterize(mu, logvar)
z.sum().backward()
print("z:", z.detach(), "| ∂Σz/∂mu:", mu.grad, "| ∂Σz/∂logvar:", logvar.grad)
ck.reparameterize(reparameterize)

### 实现 VAE（照 pytorch/examples/vae）

结构与官方示例逐层对应：`fc1` 784→400，`fc21`/`fc22` 400→20（分别输出 $\mu$ 与 $\log\sigma^2$），`fc3` 20→400，`fc4` 400→784 + sigmoid。损失 = BCE（对像素求和）+ KL，Adam 1e-3、batch 128。

In [ ]:
class VAE(nn.Module):
    # pytorch/examples/vae/main.py 的 VAE 类（BSD-3-Clause），潜维度做成参数
    def __init__(self, z_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(784, 400)
        self.fc21 = nn.Linear(400, z_dim)      # mu
        self.fc22 = nn.Linear(400, z_dim)      # logvar
        self.fc3 = nn.Linear(z_dim, 400)
        self.fc4 = nn.Linear(400, 784)

    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def loss_function(recon_x, x, mu, logvar, beta=1.0):
    # 返回 (总损失, 重建项, KL 项)，三者都是对整个 batch 求和
    bce = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + beta * kld, bce, kld


def train_vae(X, z_dim=20, epochs=5, beta=1.0, seed=0, verbose=False):
    experiments.set_seed(seed)
    model = VAE(z_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X), batch_size=128, shuffle=True)
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for (xb,) in loader:
            xb = xb.to(device)
            recon, mu, logvar = model(xb)
            loss, _, _ = loss_function(recon, xb, mu, logvar, beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
        if verbose:
            print(f"  epoch {epoch + 1}: 训练损失 {total / len(X):.1f} / 张")
    return model.eval()


def evaluate_vae(model, X):
    # 测试集上每张图的 (BCE, KL)：和 pytorch/examples 一样，测试时也抽一次 z
    bce = kld = 0.0
    with torch.no_grad():
        for i in range(0, len(X), 1000):
            xb = X[i:i + 1000].to(device)
            recon, mu, logvar = model(xb)
            _, b, k = loss_function(recon, xb, mu, logvar)
            bce, kld = bce + b.item(), kld + k.item()
    return bce / len(X), kld / len(X)

In [ ]:
t0 = time.time()
vae = train_vae(X_train, epochs=5, verbose=True)
test_bce, test_kl = evaluate_vae(vae, X_test)
print(f"测试集每张图：BCE {test_bce:.1f} + KL {test_kl:.1f} = {test_bce + test_kl:.1f}（{time.time() - t0:.1f} 秒）")
ck.vae_test_loss(test_bce + test_kl)

**复现一次公开数字**。pytorch/examples 的默认配置是全部 6 万张训练图、10 个 epoch。这个仓库本身没有公布运行结果；我们能找到的参考数字来自 NVIDIA *Base Command Manager 10* 手册里的 PyTorch 运行示例（[Running PyTorch](https://docs.nvidia.com/base-command-manager/product-manuals/10/machine-learning/pytorchrun.html)）：在那套环境里用默认配置跑 pytorch/examples 的 `vae`，日志显示第 10 个 epoch 的测试损失是 **105.8715**。注意这是第三方环境里的**一次单种子运行**，不是 pytorch/examples 自己发布的结果。我们的代码和 pytorch/examples 逐层一致，CPU 上半分钟就能跑完——对一下数：

In [ ]:
t0 = time.time()
vae_full = train_vae(X_all, epochs=10)
full_bce, full_kl = evaluate_vae(vae_full, X_test)
print(f"全量 6 万张 × 10 epochs：测试损失 {full_bce + full_kl:.2f} / 张（NVIDIA 手册中的运行日志 105.87）| 用时 {time.time() - t0:.0f} 秒")

差不到 1%——这就是一次最小的"复现"：同样的代码、同样的设置，得到同样的数（别忘了参考值本身也只是别人的一次运行）。和 1 万张 × 5 epochs 的 130 左右相比，差距来自数据量与训练量，不是代码的问题（复现报告里"差异分析"要写的正是这种可检验的解释）。

### 从先验采样：画出从没见过的数字

VAE 训练时一直把编码往 $\mathcal{N}(0, I)$ 上拉，所以生成新图只需要：**从 $\mathcal{N}(0, I)$ 抽 $z$，交给解码器**。和 S1 里自编码器的随机解码对比（VAE 的样本大多能认出是哪个数字，但偏糊、偶尔笔画古怪——这是这个小 MLP-VAE 的真实水平；第 16–17 周的扩散模型会画得清楚得多）：

In [ ]:
with torch.no_grad():
    torch.manual_seed(0)
    samples = vae_full.decode(torch.randn(32, 20, device=device))
fig, axes = plt.subplots(1, 2, figsize=(12, 2.8))
show_images(samples, nrow=4, title="VAE: decode z ~ N(0, I) -> mostly blurry but recognisable digits", ax=axes[0])
show_images(fake[:16], nrow=4, title="Plain AE: random latent points (from S1)", ax=axes[1])
plt.tight_layout()
plt.show()

### 在两个数字之间插值

取两张测试图，用编码器的 $\mu$ 作为它们的潜表示 $z_1, z_2$，在两点之间连一条直线、等距取 10 个点，逐个解码。作为对照，下面一行是**像素空间**的直接插值（两张图按比例叠加）：

In [ ]:
def interpolate(z1, z2, n):
    # 返回 (n, d)：第 0 行是 z1，最后一行是 z2，中间等距
    t = torch.linspace(0, 1, n, device=z1.device)[:, None]
    return (1 - t) * z1 + t * z2


i1, i2 = (y_test == 1).nonzero()[0].item(), (y_test == 7).nonzero()[3].item()
i3, i4 = (y_test == 3).nonzero()[0].item(), (y_test == 8).nonzero()[0].item()
rows = []
with torch.no_grad():
    for a, b in ((i1, i2), (i3, i4)):
        mu_a, mu_b = vae_full.encode(X_test[[a, b]].to(device))[0]
        rows.append(vae_full.decode(interpolate(mu_a, mu_b, 10)).cpu())
        rows.append(interpolate(X_test[a], X_test[b], 10))
show_images(torch.cat(rows), nrow=4, title="Rows 1 and 3: latent-space interpolation; rows 2 and 4: pixel cross-fade")
plt.show()
ck.interpolate(interpolate)

潜空间插值的每一帧都"像一个数字"，笔画是连续变形过去的；像素插值只是两张图的叠影。这说明 VAE 的潜空间是**连续**的：相邻的点解码成相似的图。

### 2 维潜空间：把整个"数字流形"画出来

把潜维度设成 2，就能把潜空间整个画出来（图式参照 Keras 官方 VAE 示例）：左边是测试图的 $\mu$ 按数字着色；右边是在 $[-3, 3]^2$ 的网格上逐点解码。

In [ ]:
vae2d = train_vae(X_train, z_dim=2, epochs=5)
with torch.no_grad():
    mu2d = vae2d.encode(X_test[:3000].to(device))[0].cpu().numpy()
    g = torch.linspace(-3, 3, 15)
    grid = torch.stack(torch.meshgrid(g, -g, indexing="xy"), dim=-1).reshape(-1, 2)   # 从左上到右下
    manifold = vae2d.decode(grid.to(device))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.15, 1]})
sc = axes[0].scatter(mu2d[:, 0], mu2d[:, 1], c=y_test[:3000], cmap="tab10", s=4)
fig.colorbar(sc, ax=axes[0], ticks=range(10), label="Digit")
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)
axes[0].set_xlabel("mu_1")
axes[0].set_ylabel("mu_2")
axes[0].set_title("VAE latent means sit around the origin, no empty gaps")
show_images(manifold, nrow=15, title="Decoding a 15 x 15 grid over [-3, 3]²", ax=axes[1])
plt.tight_layout()
plt.show()

对比 S1 的自编码器潜空间：那里的坐标范围由网络随意决定（可以到十几），团块之间有空地；VAE 的 $\mu$ 被 KL 项拉到原点附近的一个圆盘里，网格上的每一点都解码成一个像样的数字，相邻格子之间平滑过渡。

---
## S3 · 迁移到 FashionMNIST，以及"似然的选择"这一受控实验

### 同一套代码换数据

代码一行不改，数据换成第 9 周的 FashionMNIST。先看 2 维潜空间，和第 6 周学过的 PCA（取前 2 个主成分）放在一起：

In [ ]:
fm_train, fm_test = load_dataset(datasets.FashionMNIST, True), load_dataset(datasets.FashionMNIST, False)
F_all, fy_all = to_pixels(fm_train)
F_test, fy_test = to_pixels(fm_test)
F_train, fy_train = F_all[:10000], fy_all[:10000]

fvae2d = train_vae(F_train, z_dim=2, epochs=5)
fb, fk = evaluate_vae(fvae2d, F_test)
print(f"FashionMNIST，z = 2：测试损失 {fb + fk:.1f} / 张（MNIST 同配置约 172——衣服的像素更'灰'，BCE 天然更大，不是 bug）")

with torch.no_grad():
    mu_tr = fvae2d.encode(F_train.to(device))[0].cpu().numpy()
    mu_te = fvae2d.encode(F_test.to(device))[0].cpu().numpy()
pca2 = PCA(n_components=2).fit(F_train.numpy())
pc_tr, pc_te = pca2.transform(F_train.numpy()), pca2.transform(F_test.numpy())
knn_vae = KNeighborsClassifier(n_neighbors=5).fit(mu_tr, fy_train.numpy()).score(mu_te, fy_test.numpy())
knn_pca = KNeighborsClassifier(n_neighbors=5).fit(pc_tr, fy_train.numpy()).score(pc_te, fy_test.numpy())
print(f"2 维特征上的 5-近邻分类准确率：VAE 的 mu {knn_vae:.3f} | PCA 前 2 个主成分 {knn_pca:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, pts, name in ((axes[0], mu_te[:3000], f"VAE latent mean (kNN acc {knn_vae:.2f})"),
                      (axes[1], pc_te[:3000], f"PCA, first 2 components (kNN acc {knn_pca:.2f})")):
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=fy_test[:3000], cmap="tab10", s=4)
    ax.set_title(name)
cbar = fig.colorbar(sc, ax=axes, ticks=range(10))
cbar.ax.set_yticklabels(fm_train.classes, fontsize=7)
fig.suptitle("In 2-D, the VAE separates clothing classes better than PCA")
plt.show()

两张图都没有用到标签，但 VAE 的 2 维潜空间里，同类衣服聚得更紧（用"5-近邻分类准确率"把这个印象变成一个数——作业里你会用同一个指标比较不同的 β）。PCA 是线性投影，只能找"方差最大的两个方向"；VAE 的编码器是非线性的。

### 似然的选择：一次受控实验

VAE 的重建项是 $-\log p(x|z)$——**选什么分布作为 $p(x|z)$，就决定了损失函数长什么样**：

| 似然 $p(x \mid z)$ | 负对数似然（重建项） | 隐含的假设 |
| --- | --- | --- |
| 伯努利（pytorch/examples 的选择） | $\mathrm{BCE}(\hat{x}, x) = -\sum [x\log\hat{x} + (1-x)\log(1-\hat{x})]$ | 每个像素是"黑/白"的概率 |
| 高斯，固定方差 $\sigma^2$ | $\frac{1}{2\sigma^2}\sum (x - \hat{x})^2 + \text{常数}$ ——即 MSE | 像素 = 解码输出 + 高斯噪声 |

把它当成一次**受控实验**（第 6 周起的规矩）：**只改似然这一个变量**，结构、数据、epoch、种子、优化器全部相同。高斯取 $\sigma^2 = 1$（"单位方差"，最常见的默认）。比较三样东西：重建误差（每像素 MSE）、重建图的**清晰度**（用拉普拉斯算子响应的方差衡量：边缘越锐利，值越大）、以及 KL。

In [ ]:
def gaussian_nll(recon_x, x, sigma2=1.0):
    # 固定方差的高斯负对数似然（省去与参数无关的常数）
    return 0.5 * torch.sum((x - recon_x) ** 2) / sigma2


def train_vae_lik(X, likelihood, sigma2=1.0, epochs=5, anneal_epochs=0, seed=0):
    # 同 train_vae，只是重建项可选；anneal_epochs > 0 时 KL 权重在前 anneal_epochs 个 epoch 内从 0 线性升到 1
    experiments.set_seed(seed)
    model = VAE(20).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X), batch_size=128, shuffle=True)
    step, anneal_steps = 0, anneal_epochs * len(loader)
    for _ in range(epochs):
        model.train()
        for (xb,) in loader:
            xb = xb.to(device)
            recon, mu, logvar = model(xb)
            rec = (F.binary_cross_entropy(recon, xb, reduction="sum") if likelihood == "bernoulli"
                   else gaussian_nll(recon, xb, sigma2))
            kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            weight = min(1.0, step / anneal_steps) if anneal_steps else 1.0
            loss = rec + weight * kld
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step += 1
    return model.eval()


LAPLACE = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]).reshape(1, 1, 3, 3)


def sharpness(images):
    # 拉普拉斯响应的方差（常用的"模糊度"指标）：越大越清晰
    return F.conv2d(images.detach().cpu().reshape(-1, 1, 28, 28), LAPLACE).var(dim=(1, 2, 3)).mean().item()


def likelihood_report(model, X):
    with torch.no_grad():
        mu, logvar = model.encode(X.to(device))
        recon = model.decode(mu).cpu()
        kl_per_dim = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean(0).cpu()
    return {"mse": F.mse_loss(recon, X).item(), "sharpness": sharpness(recon),
            "kl": kl_per_dim.sum().item(), "kl_per_dim": kl_per_dim}


X_eval = F_test[:2000]
lik_models = {"Bernoulli (BCE)": train_vae_lik(F_train, "bernoulli"),
              "Gaussian, sigma² = 1 (MSE)": train_vae_lik(F_train, "gaussian", sigma2=1.0)}
lik_stats = {name: likelihood_report(m, X_eval) for name, m in lik_models.items()}
print(f"真实图片的清晰度 {sharpness(X_eval):.4f}")
for name, s in lik_stats.items():
    print(f"{name:28s} 重建 MSE {s['mse']:.4f} | 重建清晰度 {s['sharpness']:.4f} | KL {s['kl']:.1f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 5.2))
with torch.no_grad():
    torch.manual_seed(0)
    z_prior = torch.randn(12, 20, device=device)
    show_images(X_eval[:12], nrow=1, title="Test images", ax=axes[0])
    for ax, (name, m) in zip(axes[1:], lik_models.items()):
        recon = m.decode(m.encode(X_eval[:12].to(device))[0])
        show_images(recon, nrow=1, title=f"{name}: reconstructions (sharpness {lik_stats[name]['sharpness']:.3f})", ax=ax)
fig.suptitle("Same VAE, only the likelihood changed: the Gaussian one reconstructs blurrier")
plt.tight_layout()
plt.show()

单位方差的高斯似然重建得更糊：MSE 更大、清晰度更低，KL 也小得多——编码器往潜变量里"写"的信息变少了。原因藏在系数 $\frac{1}{2\sigma^2}$ 里，但要看的是**梯度的大小**，不是损失的数值：

- 高斯重建项 $\frac{1}{2\sigma^2}(x - \hat{x})^2$ 对解码输出 $\hat{x}$ 的梯度是 $(\hat{x} - x)/\sigma^2$；
- BCE 对 $\hat{x}$ 的梯度是 $(\hat{x} - x)/[\hat{x}(1-\hat{x})]$——在局部，它就像一个方差为 $\hat{x}(1-\hat{x})$ 的高斯，而 $\hat{x}(1-\hat{x}) \le 0.25$。

所以同样的重建误差，BCE 把 $\hat{x}$ 拉回 $x$ 的力至少是 $\sigma^2 = 1$ 的高斯的 4 倍（输出接近 0 或 1 的像素还要大得多）。重建项的梯度一弱，KL 项相对就变"贵"了，模型宁可少编码一些信息。

**别拿两项损失的数值大小来判断谁更重。** FashionMNIST 上 BCE 每张图两百多，看起来比 KL 大得多，但其中约 190 nats 是灰度像素自身的熵 $\sum_i H(x_i)$：即使重建得一模一样（$\hat{x} = x$），BCE 也降不到它以下——这是一个与模型参数无关的常数下限（MNIST 的像素大多是纯黑或纯白，这个下限只有约 46）。给损失加一个常数，梯度一点不变，最优解也不变。

In [ ]:
sweep = {s2: likelihood_report(train_vae_lik(F_train, "gaussian", sigma2=s2), X_eval) for s2 in (1.0, 0.1, 0.01)}
print(f"{'':16s} 重建 MSE | 清晰度 | KL/张")
print(f"{'Bernoulli (BCE)':16s} {lik_stats['Bernoulli (BCE)']['mse']:.4f}   | {lik_stats['Bernoulli (BCE)']['sharpness']:.4f} | {lik_stats['Bernoulli (BCE)']['kl']:.1f}")
for s2, s in sweep.items():
    print(f"Gaussian σ²={s2:<4g} {s['mse']:.4f}   | {s['sharpness']:.4f} | {s['kl']:.1f}")

$\sigma^2$ 越小（等价于 β 越小），KL 越大、重建越准越清晰。"MSE 生成的图更糊"这句流行说法并不完整：糊的不是 MSE 本身，而是**你隐含选定的那个方差**。这就是把损失函数当成受控实验的价值——你得到的不只是"哪个更好"，还有"为什么"。

### 后验坍塌与 KL 退火

把 $\sigma^2$ 再调大到 2（重建项更弱，KL 更"贵"），再把 KL **按潜维度拆开**看。同时做一个对照：β 在前 2 个 epoch 从 0 线性升到 1 的"KL 退火"版本。每种跑 3 个种子：

In [ ]:
collapse_runs = {}
for name, anneal_epochs in (("Gaussian σ² = 2", 0), ("Gaussian σ² = 2 + KL annealing", 2)):
    collapse_runs[name] = [likelihood_report(train_vae_lik(F_train, "gaussian", sigma2=2.0, anneal_epochs=anneal_epochs,
                                                           seed=seed), X_eval) for seed in (0, 1, 2)]
for name, runs in collapse_runs.items():
    active = [int((r["kl_per_dim"] > 0.1).sum()) for r in runs]
    print(f"{name:31s} KL/张 {experiments.format_result([r['kl'] for r in runs], 2)} | "
          f"重建 MSE {experiments.format_result([r['mse'] for r in runs])} | 活跃维度（3 个种子）{active}")

collapse = {"Bernoulli (BCE)": lik_stats["Bernoulli (BCE)"],
            **{name: runs[0] for name, runs in collapse_runs.items()}}     # 柱状图画种子 0
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (name, s) in zip(axes, collapse.items()):
    kl_sorted = np.sort(s["kl_per_dim"].numpy())[::-1]
    active = int((kl_sorted > 0.1).sum())
    ax.bar(range(20), kl_sorted)
    ax.axhline(0.1, color="gray", ls="--", lw=1)
    ax.set_title(f"{name}\n{active} of 20 dims active, MSE {s['mse']:.4f}", fontsize=9)
    ax.set_xlabel("Latent dimension (sorted)")
axes[0].set_ylabel("KL per dimension (nats)")
fig.suptitle("Most latent dimensions collapse onto the prior when the likelihood is weak (seed 0)")
plt.tight_layout()
plt.show()

KL 接近 0 的维度，意味着那一维的 $q(z_j|x)$ 对**所有**图都等于先验 $\mathcal{N}(0, 1)$——编码器根本没往这一维写任何关于 $x$ 的信息，解码器也学会了忽略它。这叫**后验坍塌**（posterior collapse）：KL 项赢了拔河，潜变量（的一部分）"死"了。BCE 版本 20 维几乎都活着；$\sigma^2 = 2$ 时只剩少数几维。在 Bowman et al.（2016）的文本 VAE 里情况更极端：解码器是很强的 LSTM，自己就能把句子写好，结果整个 KL 在训练早期迅速归零、潜变量完全没用。

他们的补救办法是 **KL 退火**（KL cost annealing）：训练开始时 KL 权重为 0（先当普通自编码器，让编码器学会往潜变量里写东西），再慢慢升到 1。看上一格打印的 3 个种子：退火后 KL 变大（潜变量里写进了更多信息）、重建 MSE 明显下降；活跃维度通常多一两个，但不是每个种子都如此。在我们这个小 MLP 上坍塌只是"部分的"，退火的效果温和而稳定——这就是为什么要看 3 个种子的均值 ± 标准差，而不是只看一张柱状图。

---
## S4 · 复现报告

把你的复现写成 2 页的书面报告（模板 `templates/replication_report.md`），自评表见下。动笔前先按下面的结构列提纲；想练表达的话，再按这个结构录一段 6 分钟的讲解。

### 报告结构

| 篇幅（讲解时长） | 内容 | 对应自评维度 |
| --- | --- | --- |
| 1 分钟 | 论文的主张是什么；你复现的是哪张图 / 哪个表；验收口径（数值还是趋势） | 复现范围清晰（20%） |
| 1 分钟 | 原文设置 vs 你的设置：一张对照表，标出所有改动 | 复现范围清晰 |
| 2 分钟 | **结果对比**：你的图和原文的图并排；至少两个具体数字的对比（原文、你的均值 ± 标准差、差多少） | 结果对比（40%） |
| 1.5 分钟 | **差异分析**：每个明显差异给出一个可检验的假设——改什么、预期看到什么；做过的检验与结果 | 差异分析（30%） |
| 0.5 分钟 | 可复现性：notebook 能否一键运行、用时、硬件 | 可复现性（10%） |
| — | 用下面的审稿人问题自查 | — |

### 审稿人问题清单（逐条问自己）

1. 你的结果有几个种子？原文那个数是单次运行还是均值？差异比种子间的标准差大吗？
2. 原文和你的数据划分一样吗？验证集是从哪里切出来的？
3. 你改动的设置里，哪一个你认为影响最大？做过只改它一个的实验吗？
4. 原文的那条曲线是训练指标还是测试指标？你画的是同一个吗？
5. 如果再给你一周，你最想先跑哪个实验来解释差异？
6. 别人拿到你的 notebook，"运行全部"能得到报告里的每一个数吗？

### 自评表

| 维度 | 权重 | 优秀的标准 |
| --- | --- | --- |
| 复现范围清晰 | 20% | 明确说明复现了哪张图或哪个表、使用了哪些原文设置 |
| 结果对比 | 40% | 与原文数值并列；量化差异 |
| 差异分析 | 30% | 对差异原因提出可检验的假设（数据版本、划分、超参数、随机性） |
| 可复现性 | 10% | notebook 一键运行 |

本节 S2 的"全量 6 万张 × 10 epochs vs 参考值 105.87"，就是一份最小的复现报告：范围（pytorch/examples 默认配置的测试损失；参考值出自 NVIDIA Base Command Manager 10 手册里的一次运行日志，仓库本身没有公布这个数）、对比（我们的数 vs 105.87，差不到 0.5%）、差异分析（假设：差异只来自随机性——换种子、换设备（CPU / GPU）结果都会动零点几，而**参考值本身也只是一个种子的结果**；检验：多跑几个种子，看 105.87 是否落在均值 ± 2 个标准差以内）、可复现性（一格代码、CPU 半分钟以内）。

### 本周小结

- 自编码器 = 编码器 + 瓶颈 + 解码器，用重建误差自监督训练；瓶颈越宽重建越好；线性自编码器 ≈ PCA（Baldi & Hornik 1989，误差比值约 1.0，子空间几乎重合）
- 自编码器不是生成模型：潜空间里有"空地"，随机解码多半不像图
- VAE：编码器输出 $\mu$ 与 $\log\sigma^2$；损失 = 重建（BCE，伯努利负对数似然）+ KL（$-\frac{1}{2}\sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$）；它是 $-\text{ELBO}$，而 ELBO 是 $\log p(x)$ 的下界
- 重参数化：$z = \mu + \sigma\varepsilon$，把随机性挪到与参数无关的 $\varepsilon$ 上，梯度才能穿过采样
- VAE 的潜空间连续、可采样、可插值；全量配置复现了 pytorch/examples 默认配置的参考值 105.87（NVIDIA 手册中的一次运行日志）
- 似然的选择是受控实验：固定方差的高斯似然里，$\sigma^2$ 等价于 KL 权重 $\beta = 2\sigma^2$
- 后验坍塌：KL≈0 的潜维度不携带信息；KL 退火（权重从 0 升到 1）是常用补救